In [ ]:
# 4. Cyclic Sort [Medium Priority]

# The 1-to-n variant:
def cyclic_sort(arr):
    i = 0
    while i < len(arr):
        home = arr[i] - 1          # values 1..n
        if arr[i] != arr[home]:
            arr[i], arr[home] = arr[home], arr[i]
        else:
            i += 1

In [ ]:
# 4.a. Missing Number (LC 268):

def missingNumber(arr):
    n = len(arr)
    i = 0
    while i < n:
        if arr[i] < n and arr[i] != arr[arr[i]]:
            t = arr[i]
            arr[i] = arr[t]
            arr[t] = t
        else:
            i += 1
    for j in range(n):
        if arr[j] != j:
            return j
    return n

In [ ]:
# 4.b. Find All Numbers Disappeared in an Array (LC 448)

def find_disappeared(arr):
    cyclic_sort(arr)
    missing = []
    for j in range(len(arr)):
        if arr[j] != j + 1:
            missing.append(j + 1)
    return missing

In [ ]:
# 4.c. Find All Duplicates in an Array (LC 442)

In [ ]:
# 4.d. Find the Duplicate Number (LC 287)

In [ ]:
# 4.e. First Missing Positive (LC 41)

def firstMissingPositive(arr):
    n, i = len(arr), 0
    while i < n:
        home = arr[i] - 1
        if arr[i] > 0 and arr[i] <= n and arr[i] != arr[home]:
            t = arr[i]
            arr[i] = arr[home]
            arr[home] = t
        else:
            i += 1
    for j in range(n):
        if arr[j] != j + 1:
            return j + 1
    return n + 1

In [ ]:
# 4.f. Set Mismatch (LC 645)

In [ ]:
# 5. Tim Sort [Low Priority]

MIN_RUN = 32

def tim_sort(arr: list[int]) -> None:
    n = len(arr)
    stack = []  # each entry: (start, length)

    i = 0
    while i < n:
        run_start = i
        run_len = count_run(arr, i, n)
        if run_len < MIN_RUN:
            force_len = min(MIN_RUN, n - run_start)
            insertion_sort(arr, run_start, run_start + force_len, run_start + run_len)
            run_len = force_len
        stack.append((run_start, run_len))
        merge_collapse(arr, stack)
        i = run_start + run_len

    while len(stack) >= 2:
        merge_adjacent(arr, stack, with_bottom=False)


def count_run(arr, lo, hi):
    if lo + 1 >= hi:
        return 1
    run_hi = lo + 1
    if arr[run_hi] < arr[lo]:
        while run_hi < hi and arr[run_hi] < arr[run_hi - 1]:
            run_hi += 1
        arr[lo:run_hi] = arr[lo:run_hi][::-1]
    else:
        while run_hi < hi and arr[run_hi] >= arr[run_hi - 1]:
            run_hi += 1
    return run_hi - lo


def insertion_sort(arr, lo, hi, start):
    if start == lo:
        start += 1
    while start < hi:
        pivot = arr[start]
        left, right = lo, start
        while left < right:
            mid = (left + right) // 2
            if pivot < arr[mid]:
                right = mid
            else:
                left = mid + 1
        arr[left + 1:start + 1] = arr[left:start]
        arr[left] = pivot
        start += 1


def merge_collapse(arr, stack):
    while len(stack) >= 2:
        if len(stack) >= 3:
            (bs, bl), (ms, ml), (ts, tl) = stack[-3], stack[-2], stack[-1]
            if bl <= ml + tl:
                if bl < tl:
                    merge_at(arr, stack, len(stack) - 3)
                else:
                    merge_at(arr, stack, len(stack) - 2)
                continue
        (ms, ml), (ts, tl) = stack[-2], stack[-1]
        if ml <= tl:
            merge_at(arr, stack, len(stack) - 2)
            continue
        break


def merge_adjacent(arr, stack, with_bottom):
    merge_at(arr, stack, len(stack) - 2)


def merge_at(arr, stack, i):
    s1, l1 = stack[i]
    s2, l2 = stack[i + 1]
    merge(arr, s1, s1 + l1, s2 + l2)
    stack[i] = (s1, l1 + l2)
    del stack[i + 1]


def merge(arr, lo, mid, hi):
    left = arr[lo:mid]
    i, j, k = 0, mid, lo
    while i < len(left) and j < hi:
        if arr[j] < left[i]:        # strict < for stability
            arr[k] = arr[j]
            j += 1
        else:
            arr[k] = left[i]
            i += 1
        k += 1
    while i < len(left):
        arr[k] = left[i]
        i += 1
        k += 1

In [ ]:
# 6. External Sort

import heapq
from typing import List

def external_sort(input_data: List[int], chunk_size: int) -> List[int]:
    # ---- Phase 1: split-and-sort ----
    runs = []
    for i in range(0, len(input_data), chunk_size):
        chunk = sorted(input_data[i:i + chunk_size])
        runs.append(chunk)

    # ---- Phase 2: k-way merge ----
    return merge_runs(runs)


def merge_runs(runs: List[List[int]]) -> List[int]:
    heap = []
    for r, run in enumerate(runs):
        if run:
            heapq.heappush(heap, (run[0], r, 0))

    out = []
    while heap:
        value, run_idx, pos = heapq.heappop(heap)
        out.append(value)
        if pos + 1 < len(runs[run_idx]):
            heapq.heappush(heap, (runs[run_idx][pos + 1], run_idx, pos + 1))
    return out